In [7]:
import os
os.environ["GOOGLE_API_KEY"] = "A-FnlJUzQ"

In [8]:
!pip install langchain chromadb openai tiktoken pypdf langchain_google_genai langchain-community

In [19]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

In [21]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [25]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [30]:
vector_store = Chroma(
    embedding_function=GoogleGenerativeAIEmbeddings(model='gemini-embedding-001'),
    persist_directory='my_chroma_db',
    collection_name='sample'
)

In [31]:
# add documents
vector_store.add_documents(docs)

['26c92eb9-6dde-40ad-ad77-b8f4eac874fb',
 'e225ca35-04c8-423c-88f3-2a52ad45b5a4',
 'a4dd7e03-bca7-461f-8c05-3ad69ad1fa13',
 'cdf649bc-cf7e-48d3-9b75-77c57adb2931',
 'fb7ec1e8-d4e9-4157-b10d-b3c46e191d92']

In [32]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['26c92eb9-6dde-40ad-ad77-b8f4eac874fb',
  'e225ca35-04c8-423c-88f3-2a52ad45b5a4',
  'a4dd7e03-bca7-461f-8c05-3ad69ad1fa13',
  'cdf649bc-cf7e-48d3-9b75-77c57adb2931',
  'fb7ec1e8-d4e9-4157-b10d-b3c46e191d92'],
 'embeddings': array([[-0.00982054,  0.02545763,  0.02402782, ...,  0.01414876,
         -0.01560954, -0.00266117],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca

In [33]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

In [34]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6406819820404053),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.660536527633667)]

In [43]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query=" ",
    filter={"team": "Chennai Super Kings"}
)

[(Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.7655057311058044),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.779556393623352)]

In [44]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='26c92eb9-6dde-40ad-ad77-b8f4eac874fb', document=updated_doc1)


In [45]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['26c92eb9-6dde-40ad-ad77-b8f4eac874fb',
  'e225ca35-04c8-423c-88f3-2a52ad45b5a4',
  'a4dd7e03-bca7-461f-8c05-3ad69ad1fa13',
  'cdf649bc-cf7e-48d3-9b75-77c57adb2931',
  'fb7ec1e8-d4e9-4157-b10d-b3c46e191d92'],
 'embeddings': array([[-0.00719311,  0.02029932,  0.02837158, ...,  0.01391   ,
         -0.01240376, -0.00345245],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]]),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a sin

In [46]:
# delete document
vector_store.delete(ids=['26c92eb9-6dde-40ad-ad77-b8f4eac874fb'])

In [47]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['e225ca35-04c8-423c-88f3-2a52ad45b5a4',
  'a4dd7e03-bca7-461f-8c05-3ad69ad1fa13',
  'cdf649bc-cf7e-48d3-9b75-77c57adb2931',
  'fb7ec1e8-d4e9-4157-b10d-b3c46e191d92'],
 'embeddings': array([[-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]]),
 'documents': ["Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah is considered one 